In [1]:
import xml.etree.ElementTree as ET
from collections import defaultdict
import pandas as pd

In [2]:
tree = ET.parse("/n/groups/patel/sivateja/full_drug_bank_database.xml")
root = tree.getroot()

print("ROOT TAG:")
print(root.tag)

print("\nFIRST-LEVEL CHILD TAGS:")
for child in list(root):
    print(child.tag)


# Takes 1 miute 21.8 seconds to run

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7fcf4ce5fa90>>
Traceback (most recent call last):
  File "/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


ROOT TAG:
{http://www.drugbank.ca}drugbank

FIRST-LEVEL CHILD TAGS:
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{htt

In [3]:
for child in list(root)[:5]:
    print(child.tag)

{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug


In [4]:
first_drug = list(root)[0]

print("FIELDS UNDER <drug>:")
for child in first_drug:
    print(child.tag)

FIELDS UNDER <drug>:
{http://www.drugbank.ca}drugbank-id
{http://www.drugbank.ca}drugbank-id
{http://www.drugbank.ca}drugbank-id
{http://www.drugbank.ca}name
{http://www.drugbank.ca}description
{http://www.drugbank.ca}cas-number
{http://www.drugbank.ca}unii
{http://www.drugbank.ca}state
{http://www.drugbank.ca}groups
{http://www.drugbank.ca}general-references
{http://www.drugbank.ca}synthesis-reference
{http://www.drugbank.ca}indication
{http://www.drugbank.ca}pharmacodynamics
{http://www.drugbank.ca}mechanism-of-action
{http://www.drugbank.ca}toxicity
{http://www.drugbank.ca}metabolism
{http://www.drugbank.ca}absorption
{http://www.drugbank.ca}half-life
{http://www.drugbank.ca}protein-binding
{http://www.drugbank.ca}route-of-elimination
{http://www.drugbank.ca}volume-of-distribution
{http://www.drugbank.ca}clearance
{http://www.drugbank.ca}classification
{http://www.drugbank.ca}salts
{http://www.drugbank.ca}synonyms
{http://www.drugbank.ca}products
{http://www.drugbank.ca}internationa

In [5]:
from collections import defaultdict
import joblib
import os

model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

predictive_genes = set()
predictive_genes_by_ct = defaultdict(set)

for ct in cell_types:
    split_counts = defaultdict(int)

    for split in range(1, 6):
        model_path = os.path.join(
            model_dir, ct, f"split_{split}", "maximal_classifier.joblib"
        )
        model = joblib.load(model_path)

        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                split_counts[g] += 1

    for g, count in split_counts.items():
        if count >= 2:
            predictive_genes.add(g)
            predictive_genes_by_ct[ct].add(g)

print(f"Total predictive genes (≥2 splits): {len(predictive_genes)}")

Total predictive genes (≥2 splits): 1849


In [17]:
# # predictive gene -> drugs
# gene_to_drugs = defaultdict(set)

# for drug in root.findall(".//{*}drug"):
#     drug_name = drug.findtext("{*}name", default="")

#     targets = drug.find("{*}targets")
#     if targets is None:
#         continue

#     for target in targets.findall("{*}target"):
#         if target.findtext("{*}organism", default="") != "Humans":
#             continue

#         for poly in target.findall("{*}polypeptide"):
#             gene = poly.findtext("{*}gene-name", default="")
#             if gene in predictive_genes:
#                 gene_to_drugs[gene].add(drug_name)

from collections import defaultdict

# predictive gene -> drugs
gene_to_drugs = defaultdict(set)

for drug in root.findall(".//{*}drug"):
    drug_name = drug.findtext("{*}name", default="")

    # ------------------
    # TARGETS
    # ------------------
    targets = drug.find("{*}targets")
    if targets is not None:
        for target in targets.findall("{*}target"):
            if target.findtext("{*}organism", default="") != "Humans":
                continue
            for poly in target.findall("{*}polypeptide"):
                gene = poly.findtext("{*}gene-name", default="")
                if gene in predictive_genes:
                    gene_to_drugs[gene].add(f"{drug_name} (target)")

    # ------------------
    # ENZYMES
    # ------------------
    enzymes = drug.find("{*}enzymes")
    if enzymes is not None:
        for enzyme in enzymes.findall("{*}enzyme"):
            if enzyme.findtext("{*}organism", default="") != "Humans":
                continue
            for poly in enzyme.findall("{*}polypeptide"):
                gene = poly.findtext("{*}gene-name", default="")
                if gene in predictive_genes:
                    gene_to_drugs[gene].add(f"{drug_name} (enzyme)")

    # ------------------
    # TRANSPORTERS
    # ------------------
    transporters = drug.find("{*}transporters")
    if transporters is not None:
        for transporter in transporters.findall("{*}transporter"):
            if transporter.findtext("{*}organism", default="") != "Humans":
                continue
            for poly in transporter.findall("{*}polypeptide"):
                gene = poly.findtext("{*}gene-name", default="")
                if gene in predictive_genes:
                    gene_to_drugs[gene].add(f"{drug_name} (transporter)")

    # ------------------
    # CARRIERS
    # ------------------
    carriers = drug.find("{*}carriers")
    if carriers is not None:
        for carrier in carriers.findall("{*}carrier"):
            if carrier.findtext("{*}organism", default="") != "Humans":
                continue
            for poly in carrier.findall("{*}polypeptide"):
                gene = poly.findtext("{*}gene-name", default="")
                if gene in predictive_genes:
                    gene_to_drugs[gene].add(f"{drug_name} (carrier)")

In [7]:
import pandas as pd

In [18]:
rows = []

for ct, genes in predictive_genes_by_ct.items():
    for g in genes:
        if g in gene_to_drugs:
            rows.append({
                "cell_type": ct,
                "gene": g,
                "n_drugs": len(gene_to_drugs[g]),
                "drugs": ", ".join(sorted(gene_to_drugs[g]))
            })

drugbank_condensed = pd.DataFrame(rows)

In [19]:
drugbank_condensed

,cell_type,gene,n_drugs,drugs
0,Ast,EDNRB,9,"Ambrisentan (target), Aprocitentan (target), B..."
1,Ast,MT1E,2,"Copper (carrier), Silver (target)"
2,Ast,TPT1,3,"Calcium Phosphate (target), Calcium citrate (t..."
3,Ast,NR3C1,78,"AZD-5423 (target), Alclometasone (target), Ald..."
4,Ast,ACSS1,1,ATP (target)
...,...,...,...,...
480,Ex,ATP8A1,1,Phosphatidyl serine (target)
481,Ex,ACTG1,2,"Artenimol (target), Copper (target)"
482,Ex,CCK,1,Camostat (target)
483,Ex,GAPDH,8,4-(2-Aminoethyl)Benzenesulfonyl Fluoride (targ...


In [20]:
drugbank_condensed.sort_values(
    ["cell_type", "n_drugs"], ascending=[True, False]
)

,cell_type,gene,n_drugs,drugs
3,Ast,NR3C1,78,"AZD-5423 (target), Alclometasone (target), Ald..."
14,Ast,SLC15A2,45,"Aminolevulinic acid (transporter), Amoxicillin..."
27,Ast,THRA,10,"Dextrothyroxine (target), Dronedarone (target)..."
0,Ast,EDNRB,9,"Ambrisentan (target), Aprocitentan (target), B..."
12,Ast,FTH1,9,"Ferric pyrophosphate citrate (target), Ferrous..."
...,...,...,...,...
417,Opc,LATS1,1,Fostamatinib (target)
419,Opc,PSAP,1,Di-Stearoyl-3-Sn-Phosphatidylethanolamine (tar...
423,Opc,PLCL1,1,Quinacrine (target)
425,Opc,NDUFA4,1,NADH (target)


In [21]:
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/DrugBank_Predictive_Targets.csv"
drugbank_condensed.to_csv(out_path, index=False)

In [7]:
import pandas as pd

# ----------------------------
# Curated "big-dog" genes from manuscript
# ----------------------------
big_dog_genes = [
    # Pan-cell / genetically anchored
    "ARL17B", "RASGEF1B", "LINGO1", "USP6NL", "EGFR",
    "ZBBX", "PNISR", "NKAIN2", "LRRTM4", "NDUFAF6", "CLU"

    # Microglia / immune-metabolic
    "IFI44L", "TNFRSF1B", "SRGN", "PLXDC2", "CD81",
    "PRKCA", "HIF1A", "PFKFB3", "ACSL1", "GFAP", "PICALM"

    # Oligodendrocyte / stress & transport
    "CRYAB", "CLDN11", "APOD", "LAMP1", "MT3",
    "FTH1", "SGCZ", "COL4A3",

    # Neuronal / subcluster-specific
    "CA4", "TIMP3", "P2RY14", "CPLX3", "TRPC3",
    "KIT", "ATP1B1", "RPL31"
]

big_dog_genes = set(g.upper() for g in big_dog_genes)

# ----------------------------
# Subset DrugBank condensed table
# ----------------------------
hits = drugbank_condensed[
    drugbank_condensed["gene"].str.upper().isin(big_dog_genes)
].copy()

# ----------------------------
# Report results
# ----------------------------
print(f"Queried {len(big_dog_genes)} high-priority predictive genes\n")

if hits.empty:
    print("❌ None of the selected genes have DrugBank target annotations.")
else:
    print(f"✅ {hits['gene'].nunique()} / {len(big_dog_genes)} genes have DrugBank targets:\n")

    for _, row in hits.sort_values("n_drugs", ascending=False).iterrows():
        print(f"Gene: {row['gene']}")
        print(f"Cell type: {row['cell_type']}")
        print(f"# Drugs: {row['n_drugs']}")
        print(f"Drugs: {row['drugs']}")
        print("-" * 60)

# ----------------------------
# Optional: clean table for supplement / figure
# ----------------------------
big_dog_table = hits[
    ["cell_type", "gene", "n_drugs", "drugs"]
].sort_values(["cell_type", "n_drugs"], ascending=[True, False])

big_dog_table

NameError: name 'drugbank_condensed' is not defined

In [15]:
from collections import Counter

interaction_types = Counter()

for drugs in gene_to_drugs.values():
    for d in drugs:
        if "(enzyme)" in d:
            interaction_types["enzyme"] += 1
        elif "(transporter)" in d:
            interaction_types["transporter"] += 1
        elif "(target)" in d:
            interaction_types["target"] += 1

interaction_types

Counter({'target': 2611, 'transporter': 214, 'enzyme': 153})

In [6]:
import pandas as pd
from collections import defaultdict
import joblib
import os

# ============================
# INPUT PATHS
# ============================
MODEL_DIR = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
DRUGBANK_CSV = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/DrugBank_Predictive_Targets.csv"

CELL_TYPES = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# ============================
# STEP 1: Load predictive genes per cell type (≥2 splits)
# EXACTLY matches UpSet logic
# ============================
predictive_genes_by_ct = defaultdict(set)

for ct in CELL_TYPES:
    gene_presence = defaultdict(int)

    for split in range(1, 6):
        model_path = os.path.join(
            MODEL_DIR, ct, f"split_{split}", "maximal_classifier.joblib"
        )
        model = joblib.load(model_path)

        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_presence[g.upper()] += 1

    predictive_genes_by_ct[ct] = {
        g for g, count in gene_presence.items() if count >= 2
    }

# ============================
# STEP 2: Load DrugBank hits
# ============================
drugbank_df = pd.read_csv(DRUGBANK_CSV)

drugbank_df["gene"] = drugbank_df["gene"].str.strip().str.upper()
drugbank_df["cell_type"] = drugbank_df["cell_type"].str.strip()

drugbank_genes_by_ct = defaultdict(set)
for _, row in drugbank_df.iterrows():
    drugbank_genes_by_ct[row["cell_type"]].add(row["gene"])

# ============================
# STEP 3: Compute coverage (gene-level, unique)
# ============================
rows = []

for ct in CELL_TYPES:
    predictive_genes = predictive_genes_by_ct.get(ct, set())
    drugbank_genes = drugbank_genes_by_ct.get(ct, set())

    n_pred = len(predictive_genes)
    n_drug = len(predictive_genes & drugbank_genes)

    rows.append({
        "cell_type": ct,
        "n_predictive_genes": n_pred,
        "n_predictive_genes_with_drugbank": n_drug,
        "percent_drugbank": 100 * n_drug / n_pred if n_pred > 0 else 0.0
    })

summary = (
    pd.DataFrame(rows)
    .sort_values("percent_drugbank", ascending=False)
)

# ============================
# PRINT ONLY
# ============================
print("\n=== DrugBank coverage of predictive genes (gene-level, ≥2 splits) ===")
print(summary.to_string(index=False, float_format="%.2f"))


=== DrugBank coverage of predictive genes (gene-level, ≥2 splits) ===
cell_type  n_predictive_genes  n_predictive_genes_with_drugbank  percent_drugbank
       Ex                 162                                48             29.63
      Ast                 175                                40             22.86
      Mic                 466                               104             22.32
       In                 108                                21             19.44
      Opc                 835                               160             19.16
      Oli                 621                               112             18.04


In [5]:
import os
import joblib
from collections import defaultdict

# === Paths ===
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

# === Cell types ===
cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

print("\nNumber of predictive genes per cell type (≥2 splits with importance > 0):\n")

for cell_type in cell_types:
    gene_presence = defaultdict(int)

    for split in range(1, 6):
        model_path = os.path.join(
            base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib"
        )
        if not os.path.exists(model_path):
            continue

        model = joblib.load(model_path)

        for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_presence[gene.upper()] += 1

    predictive_genes = {
        gene for gene, count in gene_presence.items() if count >= 2
    }

    print(f"{cell_type}: {len(predictive_genes)} predictive genes")


Number of predictive genes per cell type (≥2 splits with importance > 0):

Ast: 175 predictive genes
Mic: 466 predictive genes
In: 108 predictive genes
Oli: 621 predictive genes
Opc: 835 predictive genes
Ex: 162 predictive genes


In [9]:
import pandas as pd

# ============================
# INPUTS
# ============================
DRUGBANK_CSV = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/DrugBank_Predictive_Targets.csv"

# ============================
# Load DrugBank condensed table
# Expected columns: cell_type, gene, n_drugs, drugs
# ============================
drugbank_condensed = pd.read_csv(DRUGBANK_CSV)

# Clean strings
drugbank_condensed["cell_type"] = drugbank_condensed["cell_type"].astype(str).str.strip()
drugbank_condensed["gene"] = drugbank_condensed["gene"].astype(str).str.strip()
drugbank_condensed["gene_upper"] = drugbank_condensed["gene"].str.upper()

# ============================
# Curated genes explicitly emphasized / repeatedly mentioned in manuscript
# (kept conservative; no "random" additions)
# ============================
big_dog_genes = [
    # Core nominated / highlighted predictors (your prior list)
    "ARL17B", "RASGEF1B", "LINGO1", "USP6NL", "EGFR", "CLU",
    "CRYAB", "IFI44L", "TNFRSF1B", "PLXDC2", "CD81",

    # Canonical AD genes explicitly called out in-text with emphasis
    "APOE", "BIN1", "PICALM", "ABCA1", "MAPT",

    # Recurrently trajectory-influencing genes explicitly listed
    "NKAIN2", "LRP1B", "HSP90AA1", "FTH1", "SGCZ", "GFAP",

    # Additional explicitly mentioned in-text examples (genetic colocalization / GWAS overlap context)
    "NDUFAF6", "RASGEF1C",
]

big_dog_genes = {g.upper() for g in big_dog_genes}

# ============================
# Subset DrugBank condensed table to these genes
# (note: condensed table already has 1 row per (cell_type, gene) with n_drugs aggregated)
# ============================
hits = drugbank_condensed[drugbank_condensed["gene_upper"].isin(big_dog_genes)].copy()

print(f"Queried {len(big_dog_genes)} manuscript-emphasized genes\n")

if hits.empty:
    print("❌ None of the selected genes have DrugBank annotations in DrugBank_Predictive_Targets.csv")
else:
    print(f"✅ {hits['gene_upper'].nunique()} / {len(big_dog_genes)} genes have DrugBank annotations:\n")

    for _, row in hits.sort_values("n_drugs", ascending=False).iterrows():
        print(f"Gene: {row['gene']}")
        print(f"Cell type: {row['cell_type']}")
        print(f"# Drugs: {row['n_drugs']}")
        print(f"Drugs: {row['drugs']}")
        print("-" * 60)

# Clean table to paste into supplement if needed
big_dog_table = (
    hits[["cell_type", "gene", "n_drugs", "drugs"]]
    .sort_values(["cell_type", "n_drugs", "gene"], ascending=[True, False, True])
)

big_dog_table

Queried 24 manuscript-emphasized genes

✅ 8 / 24 genes have DrugBank annotations:

Gene: HSP90AA1
Cell type: Mic
# Drugs: 57
Drugs: (3E)-3-[(phenylamino)methylidene]dihydrofuran-2(3H)-one (target), (5E,7S)-2-amino-7-(4-fluoro-2-pyridin-3-ylphenyl)-4-methyl-7,8-dihydroquinazolin-5(6H)-one oxime (target), 2-(1H-pyrrol-1-ylcarbonyl)benzene-1,3,5-triol (target), 2-AMINO-4-(2,4-DICHLOROPHENYL)-N-ETHYLTHIENO[2,3-D]PYRIMIDINE-6-CARBOXAMIDE (target), 2-[(2-methoxyethyl)amino]-4-(4-oxo-1,2,3,4-tetrahydro-9H-carbazol-9-yl)benzamide (target), 2-amino-4-[2,4-dichloro-5-(2-pyrrolidin-1-ylethoxy)phenyl]-N-ethylthieno[2,3-d]pyrimidine-6-carboxamide (target), 3,6-DIAMINO-5-CYANO-4-(4-ETHOXYPHENYL)THIENO[2,3-B]PYRIDINE-2-CARBOXAMIDE (target), 3-({2-[(2-AMINO-6-METHYLPYRIMIDIN-4-YL)ETHYNYL]BENZYL}AMINO)-1,3-OXAZOL-2(3H)-ONE (target), 4-(1,3-Benzodioxol-5-Yl)-5-(5-Ethyl-2,4-Dihydroxyphenyl)-2h-Pyrazole-3-Carboxylic Acid (target), 4-(1h-Imidazol-4-Yl)-3-(5-Ethyl-2,4-Dihydroxy-Phenyl)-1h-Pyrazole (target),

,cell_type,gene,n_drugs,drugs
12,Ast,FTH1,9,"Ferric pyrophosphate citrate (target), Ferrous..."
6,Ast,ABCA1,6,"ATP (target), Glyburide (target), Probucol (ta..."
459,Ex,HSP90AA1,57,(3E)-3-[(phenylamino)methylidene]dihydrofuran-...
457,Ex,FTH1,9,"Ferric pyrophosphate citrate (target), Ferrous..."
452,Ex,CLU,6,"Copper (target), Custirsen (target), Zinc (tar..."
460,Ex,MAPT,5,"Astemizole (target), Flortaucipir (target), Fl..."
155,In,HSP90AA1,57,(3E)-3-[(phenylamino)methylidene]dihydrofuran-...
53,Mic,HSP90AA1,57,(3E)-3-[(phenylamino)methylidene]dihydrofuran-...
87,Mic,FTH1,9,"Ferric pyrophosphate citrate (target), Ferrous..."
116,Mic,APOE,8,"Copper (target), Infigratinib (carrier), Sirol..."
